# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [ ]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

In [ ]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

In [1]:
# @title User input needed.

# TODO: Fill in documents in the format mentioned in the first example.
DOCUMENTS = {
    "sample_protocol": (
        "Students must fill in documents in this section.",
        "Documents can be related to any topic - sports, manga, food, music etc.",
        "Have a minimum of 4 sentences with each document.",
        "Duplicate this for a minimum of 4 documents to test your agent."
    ),

    "fill_in1": (
        "Cricket is a popular sport played between two teams.",
        "Each team has eleven players on the field.",
        "Virat Kohli is one of the most famous cricket players.",
        "The game has three main formats called Test, ODI and T20."
    ),

    "fill_in2": (
        "Java is a popular object oriented programming language.",
        "It supports concepts such as inheritance and polymorphism.",
        "Java programs run on the Java Virtual Machine.",
        "Java is widely used for backend and enterprise development."
    ),

    "fill_in3": (
        "Artificial intelligence allows machines to perform intelligent tasks.",
        "AI systems can understand language and recognize patterns.",
        "Machine learning is an important part of artificial intelligence.",
        "AI is used in healthcare, education and many other industries."
    ),

    "fill_in4": (
        "Machine learning enables computers to learn from data.",
        "Supervised learning uses labelled training data.",
        "Unsupervised learning discovers patterns in unlabelled data.",
        "Machine learning is used in recommendation systems and prediction."
    )
}

In [4]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
      if isinstance(content, tuple):
        content = " ".join(content)
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("minimum of 4"))


[Source: sample_protocol]
Students must fill in documents in this section. Documents can be related to any topic - sports, manga, food, music etc. Have a minimum of 4 sentences with each document. Duplicate this for a minimum of 4 documents to test your agent.

[Source: fill_in3]
Artificial intelligence allows machines to perform intelligent tasks. AI systems can understand language and recognize patterns. Machine learning is an important part of artificial intelligence. AI is used in healthcare, education and many other industries.


In [13]:
# @title Ask gemini to search for something
import os
import google.generativeai as genai
import time

from google.colab import userdata
genai.configure(api_key=userdata.get("GOOGLE_API_KEY"))

model = genai.GenerativeModel(
    # This will break.
    # TODO: Find the right model to call in this cell from developer docs.
    model_name="gemini-2.5-flash",
    tools=[search_documents],
    system_instruction=(
        '''
        You are a chef training at a Michelin star restaurant.
        When you need specific information about temperatures,
        ingredients to use, cook time etc. use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        '''
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-2.5-flash


In [6]:
!pip install -q google-generativeai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-clo

In [23]:
import time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GOOGLE_API_KEY")
)

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=contents,
                config=config
            )

        except Exception as e:
            if ("429" in str(e) or "503" in str(e)) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"Server busy. Retrying in {wait} seconds...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Server busy. Retrying in 20 seconds...
Server busy. Retrying in 40 seconds...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
response = send_with_retry("Say Hello")
print(response.text)

In [ ]:
response = send_with_retry("Say Hello")
print(response.text)

In [20]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello"
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "Add a question here.",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


In [ ]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)